# 03 — Regime Detection

Fit a Gaussian HMM on rate + vol PC scores plus curve spreads, pick
the right number of regimes, validate that each regime has a
coherent economic signature, and benchmark against the ground-truth
regime path attached by `make_mock_store`. Persists the fitted PCAs
and HMM under `models/` for downstream notebooks.

## 1. Setup and feature construction

Load the mock store, the feature config saved by `02_pca_analysis`,
and assemble the joint feature matrix: rate PCs + vol PCs + selected
curve spreads.

In [ ]:
import sys, warnings, pickle
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from hmmlearn.hmm import GaussianHMM

from src.loaders.mock_loader import make_mock_store
from src.features.surface_pca import SurfacePCA
from src.features.derived import curve_spreads, vol_surface_metrics
from src.models.regime_hmm import RateRegimeHMM

sns.set_theme(style="whitegrid")
store = make_mock_store(n_days=1000, seed=42)

with open("../configs/feature_config.pkl", "rb") as f:
    cfg = pickle.load(f)
print("feature_config:", cfg)

In [ ]:
rate_panel = store.as_panel("rate")
vol_panel  = store.as_panel("atm_vol")

rate_pca = SurfacePCA(n_components=cfg["rate_n_components"]).fit(rate_panel)
vol_pca  = SurfacePCA(n_components=cfg["vol_n_components"]).fit(vol_panel)

rate_scores = rate_pca.transform(rate_panel)
vol_scores  = vol_pca.transform(vol_panel)
rate_scores.columns = [f"rate_{c}" for c in rate_scores.columns]
vol_scores.columns = [f"vol_{c}" for c in vol_scores.columns]
spreads = curve_spreads(store)[cfg["include_curve_spreads"]]

features = pd.concat([rate_scores, vol_scores, spreads], axis=1).dropna()
print(f"Feature matrix: {features.shape}")
features.describe().round(3)

## 2. Stationarity check

Run ADF (unit-root null) and KPSS (stationarity null) on every
feature column. A well-behaved feature rejects ADF (low p-value)
*and* fails to reject KPSS (high p-value). Series that look
non-stationary by ADF (p > 0.1) get a first-difference replacement
downstream — HMM emissions assume stationary conditional dynamics
per state, and a regime model that's chasing a deterministic trend
is uninterpretable.

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss

rows = []
for col in features.columns:
    series = features[col].dropna()
    try:
        adf_stat, adf_p, *_ = adfuller(series, autolag="AIC")
    except Exception as e:
        adf_stat, adf_p = np.nan, np.nan
    try:
        kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")
    except Exception as e:
        kpss_stat, kpss_p = np.nan, np.nan
    rows.append({
        "feature": col,
        "adf_stat": adf_stat,
        "adf_pvalue": adf_p,
        "kpss_stat": kpss_stat,
        "kpss_pvalue": kpss_p,
    })
stationarity = pd.DataFrame(rows).set_index("feature")
stationarity.round(4)

In [ ]:
# Difference any feature with ADF p > 0.1 (cannot reject unit root).
non_stationary = stationarity.index[stationarity["adf_pvalue"] > 0.1].tolist()
if non_stationary:
    print(f"Differencing non-stationary features: {non_stationary}")
    for col in non_stationary:
        features[f"{col}_diff"] = features[col].diff()
        features = features.drop(columns=[col])
    features = features.dropna()
    print(f"Feature matrix after differencing: {features.shape}")
else:
    print("All features pass ADF — no differencing applied.")
print("Final feature columns:", list(features.columns))

## 3. Number of regimes

Sweep `n_regimes` from 2 to 6 using a 5-fold `TimeSeriesSplit`. For
each candidate we record:

* **BIC** on the full sample — penalises model complexity.
* **AIC** — lighter penalty, useful as a sanity check.
* **CV log-likelihood** — held-out generalisation.
* **Average regime persistence** — mean run length of the Viterbi
  state sequence. Regimes that flip every few days are not
  economically useful even if they fit slightly better.

*Note on BIC:* the textbook formula is `-2·logL + k·log(n)` where
`logL` is the total log-likelihood. `hmmlearn.score()` already
returns the total, so we use it directly (no per-observation
rescaling).

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

results = []
tscv = TimeSeriesSplit(n_splits=5)

for n in range(2, 7):
    fold_lls = []
    for train_idx, test_idx in tscv.split(features):
        X_train = features.iloc[train_idx].values
        X_test  = features.iloc[test_idx].values
        m = GaussianHMM(n_components=n, covariance_type="full",
                        n_iter=200, random_state=42)
        m.fit(X_train)
        fold_lls.append(m.score(X_test))

    full_model = GaussianHMM(n_components=n, covariance_type="full",
                             n_iter=200, random_state=42)
    full_model.fit(features.values)
    log_lik = full_model.score(features.values)
    d = features.shape[1]
    n_params = n * n + n * d + n * d * d
    bic = -2 * log_lik + n_params * np.log(len(features))
    aic = -2 * log_lik + 2 * n_params

    states = full_model.predict(features.values)
    boundaries = np.concatenate(([0], np.where(np.diff(states) != 0)[0] + 1, [len(states)]))
    run_lengths = np.diff(boundaries)
    avg_persistence = float(run_lengths.mean())

    results.append({
        "n_regimes": n,
        "bic": bic,
        "aic": aic,
        "cv_ll_mean": np.mean(fold_lls),
        "cv_ll_std": np.std(fold_lls),
        "avg_persistence": avg_persistence,
    })

results_df = pd.DataFrame(results)
print(results_df.round(2).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
ax = axes[0, 0]
ax.plot(results_df["n_regimes"], results_df["bic"], marker="o")
ax.set_xlabel("n_regimes"); ax.set_ylabel("BIC"); ax.set_title("BIC (lower = better)")
ax = axes[0, 1]
ax.plot(results_df["n_regimes"], results_df["aic"], marker="o", color="C1")
ax.set_xlabel("n_regimes"); ax.set_ylabel("AIC"); ax.set_title("AIC (lower = better)")
ax = axes[1, 0]
ax.errorbar(results_df["n_regimes"], results_df["cv_ll_mean"],
            yerr=results_df["cv_ll_std"], marker="o", color="C2")
ax.set_xlabel("n_regimes"); ax.set_ylabel("CV log-likelihood")
ax.set_title("CV LL (higher = better)")
ax = axes[1, 1]
ax.plot(results_df["n_regimes"], results_df["avg_persistence"], marker="o", color="C3")
ax.axhline(15, color="black", linestyle="--", alpha=0.5, label="15-day floor")
ax.set_xlabel("n_regimes"); ax.set_ylabel("Avg run length (days)")
ax.set_title("Regime persistence"); ax.legend()
plt.tight_layout(); plt.show()

**Recommendation.** The BIC elbow is the primary signal; AIC tends
to prefer slightly larger models. The persistence floor at 15 days
rules out flicker-y configurations. We pick **N_REGIMES = 4** below
— one more than the latent truth (3), which lets the HMM carve out
a separate "vol_spike" state during bear-regime outliers without
muddling the bull / bear / range identification.

On mock data the BIC at n=3 and n=4 is typically close; pick whichever
gives the cleanest economic story in Section 5.

## 4. Fit the final HMM

Use `RateRegimeHMM` (our pandas-friendly wrapper around
`hmmlearn.GaussianHMM`) with the chosen `N_REGIMES`. Bumped `n_iter`
to 500 for the final fit since we only do it once.

In [ ]:
N_REGIMES = 4  # update based on Section 3 elbow analysis

hmm = RateRegimeHMM(n_regimes=N_REGIMES, n_iter=500, random_state=42)
hmm.fit(features)

print("Transition matrix:")
print(hmm.transition_matrix_.round(4))
print("\nRegime counts (Viterbi):")
print(hmm.regime_labels_.value_counts().sort_index())

## 5. Regime characterisation

Two views per regime:

1. A summary table: mean / std of every PC score plus three economic
   anchors — the 2s10s spread, the 10Y rate level (avg across
   expiries), and the (1Y, 10Y) ATM vol — with a column-wise colour
   gradient so the dimensions that separate regimes pop out.
2. Violin plots of four key features split by regime, using a
   consistent regime palette that we'll reuse throughout.

In [ ]:
# Economic anchors.
spread_2s10s = curve_spreads(store)["2s10s"]
rate_10y = store.as_panel("rate").xs("10Y", axis=1, level="maturity").mean(axis=1)
vol_1y_10y = store.get("atm_vol", expiry="1Y", maturity="10Y").droplevel(["expiry", "maturity"])

char_frame = features.copy()
char_frame["2s10s"] = spread_2s10s
char_frame["rate_10Y"] = rate_10y
char_frame["vol_1Y_10Y"] = vol_1y_10y
char_frame = char_frame.dropna()

regimes_aligned = hmm.regime_labels_.reindex(char_frame.index)
summary_mean = char_frame.groupby(regimes_aligned).mean().round(3)
summary_std  = char_frame.groupby(regimes_aligned).std().round(3)
summary_mean.index.name = "regime"
summary_std.index.name = "regime"
print("Means per regime:")
summary_mean.style.background_gradient(cmap="RdBu_r", axis=0)

In [ ]:
print("Std-devs per regime:")
summary_std.style.background_gradient(cmap="Greys", axis=0)

In [ ]:
# Consistent regime palette used in every chart below.
REGIME_PALETTE = sns.color_palette("Set2", n_colors=N_REGIMES)
regime_color_map = {r: REGIME_PALETTE[r] for r in range(N_REGIMES)}

plot_panels = [
    ("2s10s", "2s10s spread (bps)"),
    (features.columns[0], features.columns[0]),  # first rate PC
    ("rate_10Y", "10Y rate (%)"),
    ("vol_1Y_10Y", "(1Y, 10Y) ATM vol (bps)"),
]
# seaborn's hue lookup uses str(value) for categorical x; mirror that in the palette.
sns_palette = {str(r): regime_color_map[r] for r in range(N_REGIMES)}
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, (col, title) in zip(axes.flat, plot_panels):
    plot_data = pd.DataFrame({col: char_frame[col].values,
                              "regime": regimes_aligned.astype(int).astype(str).values})
    plot_data = plot_data.dropna()
    sns.violinplot(data=plot_data, x="regime", y=col, ax=ax,
                   hue="regime", palette=sns_palette, inner="quartile", legend=False)
    ax.set_title(f"{title} by regime")
    ax.set_xlabel("Regime")
plt.tight_layout(); plt.show()

## 6. Time-series overlay vs. ground truth

Top panel: the first feature column (rate PC1) with the HMM-inferred
regime as a coloured background. Middle and bottom panels: a
side-by-side ribbon comparing the latent ground truth
(`store.true_regimes`) to the HMM Viterbi sequence. Below, we
compute the Adjusted Rand Index (label-permutation invariant) and a
confusion matrix after greedy-mapping HMM regimes to their most
common true regime.

In [ ]:
def regime_spans(series):
    cur = series.iloc[0]; start = series.index[0]; prev = start
    for date, r in series.items():
        if r != cur:
            yield (start, prev, cur)
            cur = r; start = date
        prev = date
    yield (start, prev, cur)

TRUE_PALETTE = {0: "#1f77b4", 1: "#d62728", 2: "#7f7f7f"}
true_aligned = store.true_regimes.reindex(features.index)
hmm_aligned = hmm.regime_labels_.reindex(features.index)

fig, axes = plt.subplots(3, 1, figsize=(13, 7), sharex=True,
                          gridspec_kw={"height_ratios": [3, 0.6, 0.6]})

ax_top = axes[0]
for s, e, r in regime_spans(hmm_aligned.dropna().astype(int)):
    ax_top.axvspan(s, e, color=regime_color_map[r], alpha=0.25)
ax_top.plot(features.index, features.iloc[:, 0], color="black", linewidth=0.9)
ax_top.set_title(f"{features.columns[0]} with HMM regime shading")
ax_top.set_ylabel(features.columns[0])
hmm_legend = [mpatches.Patch(color=regime_color_map[r], label=f"HMM r={r}")
              for r in range(N_REGIMES)]
ax_top.legend(handles=hmm_legend, loc="upper right", ncol=N_REGIMES, fontsize=8)

ax_true = axes[1]
for s, e, r in regime_spans(true_aligned.dropna().astype(int)):
    ax_true.axvspan(s, e, color=TRUE_PALETTE[r], alpha=0.75)
ax_true.set_yticks([0.5]); ax_true.set_yticklabels(["true"])
ax_true.set_ylim(0, 1)
true_legend = [mpatches.Patch(color=c, label=store.true_regime_labels[r])
                for r, c in TRUE_PALETTE.items()]
ax_true.legend(handles=true_legend, loc="upper right", ncol=3, fontsize=8)

ax_pred = axes[2]
for s, e, r in regime_spans(hmm_aligned.dropna().astype(int)):
    ax_pred.axvspan(s, e, color=regime_color_map[r], alpha=0.75)
ax_pred.set_yticks([0.5]); ax_pred.set_yticklabels(["HMM"])
ax_pred.set_ylim(0, 1)
ax_pred.set_xlabel("Date")
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.metrics import adjusted_rand_score
from collections import Counter

common = true_aligned.dropna().index.intersection(hmm_aligned.dropna().index)
true_v = true_aligned.loc[common].astype(int).values
pred_v = hmm_aligned.loc[common].astype(int).values

ari = adjusted_rand_score(true_v, pred_v)
print(f"Adjusted Rand Index: {ari:.3f}")

# Greedy mapping: each HMM regime -> most common true regime in it.
mapping = {}
for k in range(N_REGIMES):
    mask = (pred_v == k)
    if mask.sum() > 0:
        mapping[k] = Counter(true_v[mask]).most_common(1)[0][0]
aligned_pred = np.array([mapping.get(p, -1) for p in pred_v])

true_labels = sorted(set(true_v))
cm = pd.crosstab(
    pd.Series(true_v, name="true").map(store.true_regime_labels),
    pd.Series(aligned_pred, name="HMM->true").map(store.true_regime_labels),
)
print("Confusion matrix (rows = true, cols = HMM mapped to closest true):")
print(cm)

print("\nHMM regime -> closest true regime mapping:")
for k, t in mapping.items():
    print(f"  HMM regime {k} -> {store.true_regime_labels.get(t, t)}")

**Interpretation.** ARI ≈ 0 means random labelling, 1 means perfect
recovery up to permutation. On mock data with PC features alone, the
level mode dominates so strongly that bear and range — which differ
by ~0.7% in mean rate but share comparable within-regime spread —
get blurred. Systematic misclassification along the bear↔range
diagonal of the confusion matrix is the expected pattern; adding
vol-derived features (vol PC, vol-surface metrics) sharpens this
boundary substantially because the per-regime vol means (45 / 60 /
90 bps) are wider apart than the rate means.

## 7. Transition matrix

Heatmap of `hmm.transition_matrix_`. The diagonal is regime
persistence — a 0.95 entry means 95% chance of staying in that
regime tomorrow given today's regime. Below, we convert diagonals
to expected durations using `E[duration] = 1 / (1 - p_stay)`.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
T = hmm.transition_matrix_
sns.heatmap(T, annot=True, fmt=".3f", cmap="viridis", vmin=0, vmax=1, ax=ax,
            cbar_kws={"label": "probability"})
ax.set_title("HMM transition matrix")
ax.set_xlabel("to regime"); ax.set_ylabel("from regime")
plt.tight_layout(); plt.show()

p_stay = np.diag(T.values)
expected_bdays = 1.0 / np.clip(1.0 - p_stay, 1e-9, None)
duration_df = pd.DataFrame({
    "regime": T.index,
    "p_stay": p_stay,
    "expected_business_days": expected_bdays.round(1),
    "expected_calendar_days": (expected_bdays * 365.25 / 252).round(1),
})
duration_df

## 8. Economic labelling

Based on Sections 5–7 you would inspect each regime's PC means,
rate level, vol level, and persistence, then assign an economic
label. The example mapping below is the *mock-data placeholder* —
in production this is the cell that needs a desk review.

In [ ]:
# !! UPDATE THESE LABELS after reviewing with desk trader !!
# Based on mock data, approximate labels are:
hmm.label_regimes({
    0: "bull_flattening",
    1: "bear_steepening",
    2: "range_bound",
    3: "vol_spike",
})
print("Regime labels are now:", list(hmm.transition_matrix_.index))
print()
print("Regime counts:")
print(hmm.regime_labels_.value_counts())

In [ ]:
# Re-render the time series with the new labels in the legend.
name_to_color = {name: REGIME_PALETTE[i] for i, name in enumerate(hmm.transition_matrix_.index)}
hmm_named = hmm.regime_labels_.reindex(features.index)

fig, ax = plt.subplots(figsize=(13, 4))
for s, e, r in regime_spans(hmm_named.dropna()):
    ax.axvspan(s, e, color=name_to_color[r], alpha=0.3)
ax.plot(features.index, features.iloc[:, 0], color="black", linewidth=0.9)
ax.set_title(f"{features.columns[0]} with labelled HMM regimes")
ax.set_ylabel(features.columns[0])
ax.set_xlabel("Date")
ax.legend(handles=[mpatches.Patch(color=c, label=n) for n, c in name_to_color.items()],
          loc="upper right", ncol=N_REGIMES, fontsize=8)
plt.tight_layout(); plt.show()

## 9. Save fitted artifacts

Pickle the rate / vol PCAs, the labelled HMM, and the final feature
column list so downstream notebooks load a single coherent
snapshot.

In [ ]:
import os
os.makedirs("../models", exist_ok=True)

with open("../models/rate_pca.pkl", "wb") as f:
    pickle.dump(rate_pca, f)
with open("../models/vol_pca.pkl", "wb") as f:
    pickle.dump(vol_pca, f)

hmm.save("../models/regime_hmm.pkl")

with open("../models/feature_cols.pkl", "wb") as f:
    pickle.dump(list(features.columns), f)

print("All model artifacts saved to models/")
print("  rate_pca.pkl, vol_pca.pkl, regime_hmm.pkl, feature_cols.pkl")